# 矩阵乘法介绍

矩阵乘法是深度学习与科学计算中最核心的运算之一。在昇腾AI处理器上，Ascend C 提供了一套高效、灵活的矩阵编程接口。本文将系统介绍矩阵乘法的基础知识，包括计算原理、数据格式、编程范式等内容，为后续使用Ascend C编程实现高性能MatMul算子的完整方法提供坚实的理论依据。

本节学习大纲如下：
- 矩阵乘法基础
- 矩阵乘相关分形格式
- 矩阵编程范式

---

## 1. 矩阵乘法基础

矩阵乘法是深度学习与高性能计算中的核心运算之一, 矩阵乘（MatMul）的基本计算公式为：

C = A × B + Bias

其中：

- A（左矩阵）：形状为 [M, K]

- B（右矩阵）：形状为 [K, N]

- C（结果矩阵）：形状为 [M, N]

- Bias（偏置向量）：形状为 [N]，在推理或全连接层中常用

在 Ascend C 中，该公式通过 Matmul 对象实现，支持半精度（half）和单精度（float）数据类型。

<img src="./images/matrix_multiplication.png" alt="矩阵乘法"  width="700px" >

---

## 2. 矩阵乘相关分形格式

数据排布格式（Data Layout Format）是深度学习中对多维Tensor在内存中存储方式的描述。常见的格式包括 NHWC 和 NCHW，它们为张量的每个维度赋予了特定的语义（如批大小、通道、高度、宽度）。除了这些通用格式外，为了充分发挥 AI 计算硬件（如 Ascend AI Core 中的 Cube 计算单元）的并行计算能力，Ascend C还引入了一系列特殊的分形格式，如 FRACTAL_NZ（简称 NZ）、FRACTAL_ZZ、NC1HWC0 等。这类格式通过重塑数据在内存中的排列方式，显著提升了矩阵乘、卷积等计算密集型运算的效率。

**为什么需要分形格式？**

AI Core 中的 Cube 单元是专为矩阵运算优化的硬件模块，其计算模式并非逐元素操作，而是每次以小数据块(16×16×16)为单位进行并行计算（以half数据类型为例）。为了在一个时钟周期内高效地为计算单元提供数据，内存中的数据必须满足以下条件：
 
- **连续访问**：计算所需的数据块应尽量连续存储，以最大化内存带宽利用率。

- **数据复用**：合理安排数据布局，使已加载的数据能在多次计算中被重复使用，减少数据搬运开销。

传统的 ND（行优先/列优先）格式虽然适合 CPU 的缓存访问模式，但在面对 Cube 单元的块计算时，数据往往呈现跳跃式分布，导致访存延迟增加、效率降低。为此，分形格式通过数据重组，实现了计算数据在物理内存中的“对齐”。

如下图所示，计算时要求 A 矩阵（位于 L0A Buffer）为 FRACTAL_ZZ 格式，B 矩阵（位于 L0B Buffer）为 FRACTAL_ZN 格式，C 矩阵（位于 L0C Buffer）为 FRACTAL_NZ 格式。这些格式将矩阵划分为多个分形块，适配 Cube 计算单元每次读取 (16, 16. × (16, 16) 数据进行计算的硬件特点（以 half 为例），从而显著提升矩阵计算效率。

<img src="./images/nz_format.png" alt="NZ格式"  width="700px" >

**NZ格式的内存排布逻辑**

NZ 格式是一种针对张量Tensor最低两维（一个Tensor的所有维度，右侧为低维，左侧为高维）进行重组的分形存储格式。其核心思想是通过**填充（Pad）、拆分（Reshape）和转置（Transpose）操作**，将原始数据重新排列为适合硬件块计算的布局。

假设原始张量形状为 (..., M, N)，转换过程如下：

1. 填充
将 M 和 N 填充至分形块大小的整数倍：
(..., M, N) → (..., M1×M0, N1×N0)
其中 M0、N0 为分形块大小（如 16）。

2. 拆分维度
将填充后的两个维度拆分为分形块维度和块内维度：
(..., M1×M0, N1×N0) → (..., M1, M0, N1, N0)

3. 转置重组
调整维度顺序，形成分形布局：
(..., M1, M0, N1, N0) → (..., N1, M1, M0, N0)

**NZ格式的优势**

- **宏观连续（“N字形”大块）**

    在整体内存中，相邻待计算的分形块保持连续存储，便于顺序搬移，减少寻址开销。

- **微观对齐（“Z字形”小块）**

    在单个分形块内部，数据排列顺序与 Cube 单元计算时的数据消耗顺序完全匹配，实现计算与访存的流水线对齐。

    <img src="./images/fractal_nz_format_example.png" alt="NZ示例图"  width="150px" >

**存储示例**

假设分形大小为 2×2，原始 16 个元素按行优先（ND）存储为：

`0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15`

转换为 NZ 格式后，数据被重组为：

`0, 1, 4, 5, 8, 9, 12, 13, 2, 3, 6, 7, 10, 11, 14, 15`

这种排列使得在计算相邻分形块时，所需数据在物理内存中也保持相邻，极大提高了 Cube 单元的数据吞吐效率。

<img src="./images/ND2NZ.png" alt="ND2NZ"  width="700px" >

分形格式（如 NZ）通过硬件友好的数据重排，解决了传统 ND 格式在矩阵块计算中访存不连续、数据复用率低的问题。它不仅适应了 AI Core 的并行计算特性，也为实现高性能算子（如矩阵乘、卷积）提供了关键的内存布局基础。在 Ascend C 编程中，正确使用 FRACTAL_ZZ、FRACTAL_ZN、FRACTAL_NZ 等对应格式，是发挥硬件算力的重要一环。

---

## 3. 矩阵编程范式

Cube计算的典型数据流图如下所示：

<img src="./images/matrix_programming_paradigm.png" alt="矩阵编程范式"  width="700px" >

与矢量编程类似，矩阵编程也使用逻辑位置（TPosition）来描述数据流。Cube 编程范式中主要使用的逻辑位置定义如下：

- A1：代表设备上用于矩阵计算的逻辑内存，用于存放左矩阵，物理存储对应AI Core的L1 Buffer。
- B1：代表设备上用于矩阵计算的逻辑内存，用于存放右矩阵，物理存储对应AI Core的L1 Buffer。
- C1：代表设备上用于矩阵计算的逻辑内存，用于存放Bias（偏置）数据，物理存储对应AI Core的L1 Buffer或Unified Buffer。
- A2：代表设备上用于矩阵计算的逻辑内存，用于存放小块左矩阵（如经过分割、适配L0A Buffer容量的分块），物理存储对应AI Core的L0A Buffer。
- B2：代表设备上用于矩阵计算的逻辑内存，用于存放小块右矩阵（如经过分割、适配L0B Buffer容量的分块），物理存储对应AI Core的L0B Buffer。
- C2：代表设备上用于矩阵计算的逻辑内存，用于存放小块Bias（偏置）数据（如经过分割、适配BT Buffer容量的分块），物理存储对应AI Core的BT Buffer或L0C Buffer。
- CO1：代表设备上用于矩阵计算的逻辑内存，用于存放小块矩阵计算结果（如经过分割的矩阵计算结果分块），物理存储对应AI Core的L0C Buffer。
- CO2：代表设备上用于矩阵计算的逻辑内存，用于存放矩阵计算结果（如原始矩阵的最终计算结果），物理存储对应Global Memory或AI Core的Unified Buffer。

与矢量编程的三级流水不同，矩阵乘法采用更复杂的五级流水范式，以适应其多层次的数据拆分与计算：

1. **CopyIn任务 - 数据搬入**

    将输入矩阵A、B从Global Memory搬运到Local Memory，并转换为分形格式（NZ）。Bias数据也在此阶段搬入。

2. **Split任务 - 数据分片**

    将大矩阵拆分为多个适合Cube单元计算的小分形块，为并行计算做准备。这是矩阵编程特有的阶段。

3. **Compute任务 - 分形矩阵乘**

    核心计算阶段，使用Cube指令执行分形块矩阵乘法，并累加中间结果。

4. **Aggregate任务 - 结果聚合**

    将多个分形块的计算结果聚合为完整输出矩阵，处理偏置相加等后处理操作。

5. **CopyOut任务 - 结果搬出**

    将最终结果从Local Memory转换回ND格式并搬运到Global Memory。


由此可见，采用基础API实现矩阵乘法时，开发者需要手动处理格式转换、数据切分等底层细节；而使用Matmul高阶API则可完全屏蔽这些复杂逻辑，快速完成算子开发。为简化编程范式，**Matmul高阶API对矩阵运算的核心流程进行了抽象封装**，将其归纳为CopyIn、Compute、CopyOut三个主要阶段，如下图所示：

<img src="./images/simplified_matrix_programming_paradigm.png" alt="矩阵简化编程范式" width="700px">

各阶段与API接口的对应关系如下：

- **CopyIn阶段**：通过SetTensorA、SetTensorB、SetBias接口将输入数据搬运至片上；

- **Compute阶段**：通过Iterate接口启动矩阵乘计算；

- **CopyOut阶段**：通过GetTensorC接口将结果写回全局内存。

在实际开发中，使用Matmul高阶API实现矩阵乘算子的核函数遵循以下五个标准步骤：

1. 创建Matmul对象

    实例化Matmul模板类，并通过MatmulType指定输入输出矩阵的数据类型、存储位置与数据格式。

2. 初始化

    通过REGIST_MATMUL_OBJ宏将Matmul对象与TPipe及Tiling参数绑定，完成内部资源准备。

3. 设置左矩阵A、右矩阵B、Bias

    调用SetTensorA、SetTensorB、SetBias接口，将当前AI Core负责的左矩阵A、右矩阵B及偏置Bias的全局内存视图传递给Matmul对象。

4. 执行矩阵乘操作
    
    调用Iterate（灵活控制迭代）或IterateAll（全量计算）接口启动矩阵乘计算，并通过GetTensorC获取结果。

5. 结束矩阵乘操作

    调用End接口释放Matmul计算资源。

典型代码示例如下：

```
// 创建Matmul对象 创建对象时需要传入A、B、C、Bias的参数类型信息， 类型信息通过MatmulType来定义，包括：内存逻辑位置、数据格式、数据类型。
typedef MatmulType<TPosition::GM, CubeFormat::ND, half> aType; 
typedef MatmulType<TPosition::GM, CubeFormat::ND, half> bType; 
typedef MatmulType<TPosition::GM, CubeFormat::ND, float> cType; 
typedef MatmulType<TPosition::GM, CubeFormat::ND, float> biasType; 
Matmul<aType, bType, cType, biasType> mm; 

REGIST_MATMUL_OBJ(&pipe, GetSysWorkSpacePtr(), mm, &tiling); // 初始化
// CopyIn阶段：完成从GM到LocalMemory的搬运
mm.SetTensorA(gm_a);    // 设置左矩阵A
mm.SetTensorB(gm_b);    // 设置右矩阵B
mm.SetBias(gm_bias);    // 设置Bias
// Compute阶段：完成矩阵乘计算
while (mm.Iterate()) { 
    // CopyOut阶段：完成从LocalMemory到GM的搬运
    mm.GetTensorC(gm_c); 
}
// 结束矩阵乘操作
mm.End();
```

### 3.1 Matmul高阶API详解

Matmul高阶API封装了矩阵乘的核心流程，开发者只需调用几个关键接口即可完成数据搬移、计算和结果输出。常用接口及其功能如下：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center"><strong>API</strong></td>
  <td align="center"><strong>功能描述</strong></td>
</tr>
<tr>
  <td align="left">REGIST_MATMUL_OBJ</td>
  <td align="left">初始化Matmul对象。</td>
</tr>
<tr>
  <td align="left">SetTensorA</td>
  <td align="left">设置矩阵乘的左矩阵A。</td>
</tr>
<tr>
  <td align="left">SetTensorB</td>
  <td align="left">设置矩阵乘的右矩阵B。</td>
</tr>
<tr>
  <td align="left">SetBias</td>
  <td align="left">设置矩阵乘的Bias。</td>
</tr>
<tr>
  <td align="left">Iterate</td>
  <td align="left">每调用一次Iterate，会计算出一片baseM * baseN的C矩阵。</td>
</tr>
<tr>
  <td align="left">IterateAll</td>
  <td align="left">调用一次IterateAll，会计算出singleCoreM * singleCoreN大小的C矩阵。</td>
</tr>
<tr>
  <td align="left">GetTensorC</td>
  <td align="left">配合Iterate使用，一次Iterate后，获取一块或者两块C矩阵片，可以直接输出到GM tensor中，也可以输出到VECIN tensor中。</td>
</tr>
<tr>
  <td align="left">End</td>
  <td align="left">一个matmul计算结束时需要调用一次End函数。</td>
</tr>
<tr>
  <td align="left">SetTail</td>
  <td align="left">在不改变Tiling的情况下，重新设置本次计算的singleCoreM/singleCoreN/singleCoreK。</td>
</tr>
</table>

这些API的设计目标是将复杂的五级流水（CopyIn→Split→Compute→Aggregate→CopyOut）简化为CopyIn、Compute、CopyOut三个主要阶段，开发者无需手动管理分形块的切分与排布。

- CopyIn阶段：SetTensorA、SetTensorB、SetBias负责将数据从Global Memory搬运到Local Memory，并自动完成格式转换（如ND转NZ）。

- Compute阶段：Iterate或IterateAll触发Cube单元执行分形块矩阵乘，同时累加偏置（若已设置）。每次迭代计算一个或多个分形块，具体数量由Tiling参数决定。

- CopyOut阶段：GetTensorC获取计算结果在L0C上的地址，开发者需将其搬回Global Memory，并可根据需要转换回ND格式。

通过Tiling参数（在初始化时通过REGIST_MATMUL_OBJ绑定），开发者可以精细控制分块大小、迭代次数以及每个核负责的计算范围，从而适配不同规模的矩阵和硬件资源。

### 3.2 多核数据切分与迭代原理

在多核AI处理器上，一个矩阵乘任务会被自动切分到多个AI Core并行执行。理解数据切分方式对于优化性能至关重要。

以输入矩阵A (GM, ND, half)、矩阵B(GM, ND, half)，输出矩阵C (GM, ND, float)，无Bias场景为例，其中(GM, ND, half)表示数据存放在GM上，数据格式为ND，数据类型为half，描述Matmul高阶API典型场景的算法如下图。

<img src="./images/iterate_process.png" alt="Iterate迭代过程"  width="700px" >   

#### 3.2.1 数据在多核间的切分

为了最大化并行度，切分策略优先选择左矩阵A的行（M维度）和右矩阵B的列（N维度）进行划分，而尽可能避免切分K轴（因为切分K轴会导致核间需要对中间结果进行累加，引入数据依赖）。每个核独立计算一块输出子矩阵，互不依赖。

以输入矩阵A（形状[M, K]，GM上ND格式，half）、矩阵B（形状[K, N]，GM上ND格式，half）为例，输出矩阵C（形状[M, N]，GM上ND格式，float），无Bias。若启用8个AI Core并行执行，通常的切分方式为：

- 将A的M行切分为4份，每份行数为 SingleCoreM（即 M = 4 × SingleCoreM）

- 将B的N列切分为2份，每份列数为 SingleCoreN（即 N = 2 × SingleCoreN）

- K轴保持完整，不切分

这样每个核负责计算一个 SingleCoreM × SingleCoreN 的输出子块，所需数据为A矩阵的对应行区间和B矩阵对应的列区间。8个核覆盖整个输出矩阵（4×2网格），核间完全独立，无需通信。

#### 3.2.2 单核内的分形块迭代

每个AI Core内，由于片上存储（L1/L0）容量有限，不能一次性加载全部数据。需要将负责的矩阵块进一步切分为更小的分形块，通过多层循环完成计算。单核内的数据流与迭代过程可参考上图 Iterate迭代过程中的标注。

定义以下分块参数（由Tiling决定）：

- baseM、baseN：每次Cube计算的基本块大小（如16×16）

- baseK：每次Cube计算所需的K维块大小（如16）

- stepM、stepN：每次从GM搬运到L1（A1/B1）的步长，stepM为左矩阵在A1中缓存的buffer M方向上baseM的倍数。stepN为右矩阵在B1中缓存的buffer N方向上baseN的倍数。

- SingleCoreM、SingleCoreN、 singleCoreK：A、B、C矩阵单核内shape大小，以元素为单位。singleCoreK = K，多核处理时不对K进行切分；singleCoreM <= M；singleCoreN <= N。

- depthA1、depthB1：A1、B1中全载基本块的份数，depthA1为A1中全载baseM * baseK的份数，depthB1为B1中全载baseN * baseK的份数。

单核内的完整迭代过程如下（对应图中的mIter、kIter、nIter循环）：

1. 从GM搬运矩阵块到A1（左矩阵），按行方向循环：每次从矩阵A中搬出一个大小为 stepM × baseM x baseK 的矩阵块 a1，直至覆盖当前核所需的全部行（SingleCoreM）。同时从GM搬运矩阵块到B1（右矩阵）：按列方向循环：每次从矩阵B中搬出一个大小为 baseK × stepN x baseN的矩阵块 b1，直至覆盖当前核所需的全部列（SingleCoreN）。

2. 从A1加载数据到A2：在已搬入的 a1 块内，循环取出每个大小为 baseM × baseK 的子块 a0，加载到L0A缓冲区（A2）。从B1加载数据到B2并转置：在已搬入的 b1 块内，循环取出每个大小为 baseK × baseN 的子块，加载到L0B缓冲区（B2），并完成转置，得到 baseN × baseK 的格式（满足Cube单元对B矩阵的输入要求）。

3. 执行矩阵乘：Cube单元计算一个 a0（baseM×baseK）与一个 b0（baseN×baseK）的乘积，结果累加到L0C缓冲区（CO1）中对应的 baseM × baseN 块。

4. 将结果从CO1聚合到CO2：每计算完一个 baseM × baseN 块，将其搬移到L1或Unified Buffer中的临时结果区（CO2），该区域大小为 SingleCoreM × SingleCoreN，用于累加当前 a1×b1 的全部中间结果。

5. 重复步骤2-4，完成当前矩阵块a1×b1的整个计算：内层循环遍历 a1 中的所有 baseM 行块和 b1 中的所有 baseN 列块，以及K方向的累加（即循环 baseK 深度的累加），最终在CO2中得到 SingleCoreM × SingleCoreN 的完整结果块。

6. 将CO2结果搬出到矩阵块C：将CO2中的整个结果块写回全局内存中对应位置（C矩阵）。

7. 重复步骤1-6，完成矩阵A * B = C的计算：外层循环遍历当前核负责的所有行分块（沿M方向）和列分块（沿N方向），最终得到完整的输出子矩阵。

通过这种多级循环与缓冲区的乒乓操作，Cube单元的计算可以始终处于满载状态，同时隐藏数据搬移的延迟，实现高效计算。图中的 mIter、kIter、nIter 分别标识了沿M、K、N维度的迭代次数，而 baseM、baseN、baseK 等参数则由Tiling配置精确控制，以适配不同规模的矩阵和硬件资源。

---

## 课后练习

请根据本节课程学习内容完成以下题目进行自测。

1. 根据文中对矩阵乘法五级流水范式的描述，以下关于“Split 任务”的核心作用，哪一项是正确的？

    A. 将计算结果从 Local Memory 搬出到 Global Memory

    B. 将大矩阵拆分为适合 Cube 单元计算的小分形块，为并行计算做准备

    C. 执行分形块矩阵乘法，并累加中间结果

    D. 将输入矩阵从 Global Memory 搬运到 Local Memory 并转换格式

2. 将传统行优先（ND）格式的张量转换为 FRACTAL_NZ 分形格式时，正确的数据重组步骤顺序是？

    A. 填充 -> 拆分维度 -> 转置重组

    B. 拆分维度 -> 转置重组 -> 填充

    C. 转置重组 -> 填充 -> 拆分维度

    D. 拆分维度 -> 填充 -> 转置重组

3. 在 Ascend C 的矩阵编程范式中，以下哪个“逻辑位置”与物理存储 “L0A Buffer” 相对应，并用于存放小块左矩阵？

    A. A1

    B. A2

    C. B2

    D. CO1

4. 使用 Ascend C 的 Matmul 高阶 API 简化编程时，以下哪个接口组合正确地对应了 “CopyIn -> Compute -> CopyOut” 的完整流程？

    A. SetTensorA -> Iterate -> SetBias

    B. SetTensorB -> GetTensorC -> Iterate

    C. Iterate -> GetTensorC -> SetTensorA

    D. SetTensorA/SetTensorB/SetBias -> Iterate -> GetTensorC

5. 分形格式（如 FRACTAL_NZ）被引入 Ascend C 的主要目的是什么？

    A. 为了统一不同深度学习框架（如TensorFlow, PyTorch）的数据格式标准

    B. 主要目的是减少存储空间占用，对计算性能没有直接影响

    C. 解决传统 ND 格式在 AI Core Cube 单元块计算时访存不连续、数据复用率低的问题

    D. 为了让程序员更直观地理解和编写矩阵运算代码

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/04.02_answer.txt